# Set up

In [1]:
# ─────────────────────────────────────────────
# 1) set_experiments
# ─────────────────────────────────────────────
import os
import pandas as pd
import plotly.express as px
from transition_function_model import (
    setup_transition_function_model,
)

def set_experiments(
    name: str,
    ruta_ICE: str,
    ruta_PG: str,
    experiment_runner,                        # function that runs the experiment
    create_plot_fn,                           # 🔹 NEW argument
    output_root: str = "Model_Discussion"
):
    """
    Runs an experiment and saves results + plot.

    Parameters
    ----------
    name             : Name of the subfolder (e.g., "Case1").
    ruta_ICE         : Path to the ICE model (.h5 or folder).
    ruta_PG          : Path to the PG model (.h5 or folder).
    experiment_runner: Callable[[TransitionFunction], pd.DataFrame]
                       Function that receives 'trans_func' and returns a DataFrame.
    create_plot_fn   : Callable[[pd.DataFrame, str], plotly.graph_objects.Figure]
                       Function responsible for building the HTML plot.
    output_root      : Root folder where results will be saved.
    """

    # 1 ▸ Output folder
    folder = os.path.join(output_root, name)
    os.makedirs(folder, exist_ok=True)

    # 2 ▸ Transition model
    trans_func = setup_transition_function_model(ruta_ICE, ruta_PG)

    # 3 ▸ Run the experiment
    df = experiment_runner(trans_func)

    # 4 ▸ Save CSV
    df.to_csv(os.path.join(folder, "raw_results.csv"), index=False)

    # 5 ▸ Create and save the plot using the provided function
    fig       = create_plot_fn(df, f"{name} — Results")
    html_path = os.path.join(folder, f"traj_{name}.html")
    fig.write_html(html_path)
    
    print(f"✔ Results and plot saved in «{folder}»")

    return folder, df


2025-10-02 09:57:45.368864: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-02 09:57:47.347043: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-02 09:57:47.352087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-02 09:57:47.590438: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-02 09:57:48.181150: I tensorflow/core/platform/cpu_feature_guar

# Experiments

## Case 1:

In [ ]:
import plotly.express as px

def plot_nox_only(df, title):
    # Each second = 2 steps  →  time [s] = step / 2
    df = df.copy()
    df["time_s"] = df["step"] / 2          # 🔸 NEW column

    fig = px.line(
        df, x="time_s", y="nox",
        title=title,
        labels={"time_s": "Time [s]", "nox": "NOx [g/s]"}
    )
    fig.update_layout(hovermode="x unified")
    return fig

In [ ]:
import tensorflow as tf
import pandas as pd

def experiment_1(
    trans_func,
    steps: int = 5000*2,            # number of simulation steps
):
    """
    Case 1: Idling at 1000 rpm with 3 mg of fuel.
    Only captures and prints NOx emission over steps.
    """

    # 1) Prepare data structure
    data = {"step": [], "nox": []}

    # 2) Constant parameters (scalars!)
    ice_sp_t   = tf.constant(1000.0, dtype=tf.float32)  # ICE_Speed [rpm]
    fuelmass_t = tf.constant(3.0,    dtype=tf.float32)  # Fuelmass  [mg]
    T_amb_t    = tf.constant(298, dtype=tf.float32)  # Ambient temperature [K]
    p_amb_t    = tf.constant(1.0,    dtype=tf.float32)  # Ambient pressure [bar]

    # 3) Simulation loop
    for k in range(steps):
        # call predict_ice with scalar for each input
        torque, nox, *_ = trans_func.predict_ice(
            ice_sp_t,
            fuelmass_t,
            T_amb_t,
            p_amb_t
        )

        # Store
        data["step"].append(k)
        # nox can arrive as scalar tensor → extract value
        data["nox"].append(float(nox.numpy()))

    # 4) Create DataFrame and print
    df = pd.DataFrame(data)

    return df

In [ ]:
# Example call to obtain the first case study
folder, df_case1 = set_experiments(
    name="Case1",
    ruta_ICE="../src/models_markus/ICE_Model_Update_01",
    ruta_PG="../src/models_markus/PG_v2",
    experiment_runner=experiment_1,
    create_plot_fn=plot_nox_only 
)

🔍 Intentando cargar modelo desde: ../src/models_markus/ICE_Model_Update_01/model.h5
🔍 Intentando cargar modelo desde: ../src/models_markus/PG_v2/model.h5


/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator MinMaxScaler from version 1.3.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names



✔ Results and plot saved in «Model_Discussion/Case1»


## Case 2:

In [ ]:
import tensorflow as tf
import pandas as pd

def experiment_2(
    trans_func,
    total_steps: int = 2500*2,                              # total steps
    interval: int = 400,                                  # how often it changes
    fuelmass_profile = [3, 10, 20, 30, 40, 50, 60, 70, 5], # original profile
    ice_speed: float = 2500.0,    # ICE_Speed [rpm]
    T_amb_K: float = 298,      # Ambient temperature [K]
    p_amb_bar: float = 1.0        # Ambient pressure [bar]
):
    """
    Case 2 refined: 2500 total steps, fuelmass change every 200.
    If profile values run out, the last one is kept.
    """
    # Prepare constant tensors
    ice_sp_t = tf.constant(ice_speed, dtype=tf.float32)
    T_amb_t  = tf.constant(T_amb_K, dtype=tf.float32)
    p_amb_t  = tf.constant(p_amb_bar, dtype=tf.float32)

    data = {"step": [], "mf": [], "torque": []}

    for step in range(total_steps):
        # which profile position corresponds?
        idx = step // interval
        if idx < len(fuelmass_profile):
            mf_val = fuelmass_profile[idx]
        else:
            mf_val = fuelmass_profile[-1]

        mf_t = tf.constant(mf_val, dtype=tf.float32)

        # predict_ice returns (torque, nox, co, ...)
        torque, *_ = trans_func.predict_ice(
            ice_sp_t,
            mf_t,
            T_amb_t,
            p_amb_t
        )

        data["step"].append(step)
        data["mf"].append(mf_val)
        data["torque"].append(float(torque.numpy()))

    return pd.DataFrame(data)

In [14]:
import plotly.express as px

def plot_torque_only(df, title="ICE Torque (load-step)"):
    # 🔸 Nuevo eje en segundos
    df = df.copy()
    df["time_s"] = df["step"] / 2

    fig = px.line(
        df, x="time_s", y="torque",
        title=title,
        labels={"time_s": "Time [s]", "torque": "Torque ICE [Nm]"}
    )
    fig.update_layout(hovermode="x unified")
    return fig


In [ ]:
# Example call to obtain the first case study
folder, df_case1 = set_experiments(
    name="Case2",
    ruta_ICE="../src/models_markus/ICE_Model_Update_01",
    ruta_PG="../src/models_markus/PG_v2",
    experiment_runner = experiment_2,
    create_plot_fn    = plot_torque_only
)

🔍 Intentando cargar modelo desde: ../src/models_markus/ICE_Model_Update_01/model.h5
🔍 Intentando cargar modelo desde: ../src/models_markus/PG_v2/model.h5


/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator MinMaxScaler from version 1.3.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names



✔ Results and plot saved in «Model_Discussion/Case2»


## Case 3:

In [38]:
# ─────────────────────────────────────────────
# 1) set_experiments
# ─────────────────────────────────────────────
import os
import pandas as pd
import plotly.express as px
from transition_function_model import (
    setup_transition_function_model,
)

def set_experiments(
    name: str,
    ruta_ICE: str,
    ruta_PG: str,
    experiment_runner,                        # function that runs the experiment
    create_plot_fn,                           # 🔹 NEW argument
    output_root: str = "Model_Discussion"
):
    """
    Runs an experiment and saves results + plot.

    Parameters
    ----------
    name             : Name of the subfolder (e.g., "Case1").
    ruta_ICE         : Path to the ICE model (.h5 or folder).
    ruta_PG          : Path to the PG model (.h5 or folder).
    experiment_runner: Callable[[TransitionFunction], pd.DataFrame]
                       Function that takes 'trans_func' and returns a DataFrame.
    create_plot_fn   : Callable[[pd.DataFrame, str], plotly.graph_objects.Figure]
                       Function responsible for building the HTML plot.
    output_root      : Root folder where results will be saved.
    """

    # 1 ▸ Output folder
    folder = os.path.join(output_root, name)
    os.makedirs(folder, exist_ok=True)

    # 2 ▸ Transition model
    trans_func = setup_transition_function_model(ruta_ICE, ruta_PG, SOC_ini=0.5)

    # 3 ▸ Run the experiment
    df = experiment_runner(trans_func)

    # 4 ▸ Save CSV
    df.to_csv(os.path.join(folder, "raw_results.csv"), index=False)

    # 5 ▸ Create and save the plot using the provided function
    fig       = create_plot_fn(df, f"{name} — Results")
    html_path = os.path.join(folder, f"traj_{name}.html")
    fig.write_html(html_path)

    print(f"✔ Results and plot saved in «{folder}»")

    return folder, df


In [ ]:
import tensorflow as tf
import pandas as pd

def experiment_3(
    trans_func,
    total_steps: int = 2500*2,
    interval: int = 800,                     
    em2_torque_profile = [50, 250, 400],
    ice_speed: float = 0.0,    # ICE stopped
    fuelmass: float = 0.0,     # no fuel
    brake: float = 0.0,        # no braking
    T_amb_K: float = 298,
    p_amb_bar: float = 1.0
):
    """
    Configurable Case 3:
    - changes em2_torque level every `interval` steps
    - if values run out, the last one is kept
    """
    # constant tensors
    ice_sp_t   = tf.constant(ice_speed, dtype=tf.float32)
    fuelmass_t = tf.constant(fuelmass,   dtype=tf.float32)
    T_amb_t    = tf.constant(T_amb_K,    dtype=tf.float32)
    p_amb_t    = tf.constant(p_amb_bar,  dtype=tf.float32)
    brake_t    = tf.constant(brake,      dtype=tf.float32)

    data = {"step": [], "em2_torque": [], "soc": [], "vel": []}
    n_levels = len(em2_torque_profile)

    for step in range(total_steps):
        # calculate which index touches and if they run out, keep the last one
        idx = step // interval
        if idx >= n_levels:
            idx = n_levels - 1

        em2_val = em2_torque_profile[idx]
        em2_t   = tf.constant(em2_val, dtype=tf.float32)

        # 1) ICE (always zero)
        torque_ice, *_ = trans_func.predict_ice(
            ice_sp_t, fuelmass_t, T_amb_t, p_amb_t
        )
        # 2) PG
        vel_out, soc_pred = trans_func.predict_PG(
            ice_sp_t, em2_t, torque_ice, brake_t
        )

        data["step"].append(step)
        data["em2_torque"].append(em2_val)
        data["soc"].append(float(soc_pred.numpy()))
        data["vel"].append(float(vel_out.numpy()))

    return pd.DataFrame(data)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_soc_vel(df, title="Case 3: SoC & Car Speed vs Time"):
    """
    Interactive plot of SoC_pred (left y-axis) and Car_Speed_pred (right y-axis).
    """
    # 🔸 Convert step to seconds
    time_s = df["step"] / 2

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(x=time_s, y=df["soc"], name="SoC_pred"),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=time_s, y=df["vel"], name="Car_Speed_pred"),
        secondary_y=True
    )

    fig.update_layout(
        title=title,
        xaxis_title="Time [s]",      # 🔸 Corrected label
        hovermode="x unified"
    )
    fig.update_yaxes(title_text="SoC [fraction]",     secondary_y=False)
    fig.update_yaxes(title_text="Car Velocity [km/h]", secondary_y=True)

    return fig

In [ ]:
# Example call to obtain the first case study
folder, df_case1 = set_experiments(
    name="Case3",
    ruta_ICE="../src/models_markus/ICE_Model_Update_01",
    ruta_PG="../src/models_markus/PG_v2",
    experiment_runner = experiment_3,
    create_plot_fn    = plot_soc_vel
)

🔍 Intentando cargar modelo desde: ../src/models_markus/ICE_Model_Update_01/model.h5
🔍 Intentando cargar modelo desde: ../src/models_markus/PG_v2/model.h5


/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning:

Trying to unpickle estimator MinMaxScaler from version 1.3.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names

/home/usuaris.new/artur.aubach/Antic_RL_Cotxe/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MinMaxScaler was fitted with feature names



✔ Results and plot saved in «Model_Discussion/Case3»
